# Initial Demo

In [ ]:
"""

Steps:
1. Biometry extraction -> List[TermBin] -> quantitative HPO terms
2. Clinical indication -> reason for exam
3. Pregnancy dating -> LMP, EDD, gestational age context
4. Clinical impression -> qualitative HPO terms from free text
5. Fetal anatomy -> structured findings + HPO terms from anomalies
6. Estimated fetal weight -> SGA/AGA/LGA classification
7. Fetal ratios -> proportionality assessment
8. Phenopacket assembly -> GA4GH Phenopacket v2.0 JSON
"""

import gzip
import json
import re
from datetime import datetime, timezone
from pathlib import Path

from google.protobuf.json_format import MessageToJson, Parse
from google.protobuf.timestamp_pb2 import Timestamp
import phenopackets.schema.v2 as pps2

# ETL Extractors (biometry -> TermBins)
from prenatalppkt.etl.extractors import observer

# ETL Section Parsers (clinical metadata -> Dicts)
from prenatalppkt.etl.sections import (
   parse_clinical_indication,
   parse_pregnancy_dating,
   parse_clinical_impression,
   parse_fetal_anatomy,
   parse_estimated_fetal_weight,
   parse_fetal_ratios,
)

# HPO Concept Recognition (fenominal: text -> HPO)
from prenatalppkt.hpo import HpoParser

# Gestational Age utilities
from prenatalppkt.gestational_age import GestationalAge

print("=" * 80)
print("PRENATALPPKT ETL PIPELINE")
print("Observer JSON -> Section Parsing -> Phenopacket v2.0")
print("=" * 80)

# =============================================================================
# STEP 1: Load the fenominal HPO Concept Recognizer (text -> HPO, detects negation)
# =============================================================================
print("\n[STEP 1] Loading the fenominal HPO Concept Recognizer...")

HP_JSON_GZ = Path("tests/data/hp.json.gz")
TMP_HP_JSON = Path("/tmp/hp.json")

with gzip.open(HP_JSON_GZ, "rt", encoding="utf-8") as f_in:
   with open(TMP_HP_JSON, "w", encoding="utf-8") as f_out:
       f_out.write(f_in.read())

hpo_parser = HpoParser(hpo_json_file=str(TMP_HP_JSON))
hpo_cr = hpo_parser.get_hpo_concept_recognizer()

print(f"HPO version: {hpo_parser.get_version()}")

# =============================================================================
# STEP 2: Load Observer JSON
# =============================================================================
print("\n[STEP 2] Loading Observer JSON...")

data_path = Path("tests/data/Apple_Sally_pretty.json")
with open(data_path) as f:
   observer_data = json.load(f)

print(f"Loaded: {data_path.name}")
print(f"Fetuses: {len(observer_data.get('fetuses', []))}")

# =============================================================================
# STEP 3: Extract Biometry -> TermBins
# =============================================================================
print("\n[STEP 3] Extracting biometry measurements...")

term_bins = observer.extract(observer_data)
print(f"Extracted {len(term_bins)} TermBins")

# Helper function to parse GA from TermBin description
def parse_ga_from_description(description: str) -> tuple[int, int]:
   """Extract weeks and days from TermBin description like 'HC: 250.0 mm (42.5%) at 26w6d'"""
   match = re.search(r"at (\d+)w(\d+)d", description)
   if match:
       return int(match.group(1)), int(match.group(2))
   return 27, 0  # fallback values for XwYd

# Display TermBins - note: TermBin has description, hpo_id, hpo_label, normal, range
# NOT label, value_mm, percentile directly
for tb in term_bins:
   status = "Normal" if tb.normal else "ABNORMAL"
   print(f"    - {tb.description} [{status}]")
   print(f"      HPO: {tb.hpo_id} - {tb.hpo_label}")

# =============================================================================
# STEP 4: Parse Clinical Sections
# =============================================================================
print("\n[STEP 4] Parsing clinical sections...")

# 4a. Clinical Indication
indication = parse_clinical_indication(observer_data, "observer_json")
indication_text = indication.get('indication_text', 'N/A') or 'N/A'
print(f"\n  [4a] Clinical Indication:")
print(f"       Reason: {indication_text[:60]}...")

# 4b. Pregnancy Dating
dating = parse_pregnancy_dating(observer_data, "observer_json")
print(f"\n  [4b] Pregnancy Dating:")
print(f"       LMP: {dating.get('lmp', 'N/A')}")
print(f"       EDD: {dating.get('edd', 'N/A')}")
print(f"       GA at exam: {dating.get('ga_weeks', 'N/A')} weeks")

# 4c. Clinical Impression (with HPO extraction)
impression = parse_clinical_impression(observer_data, "observer_json", hpo_cr=hpo_cr)
impression_text = impression.get('impression_text', 'N/A') or 'N/A'
print(f"\n  [4c] Clinical Impression:")
print(f"       Text: {impression_text[:60]}...")
print(f"       HPO terms found: {len(impression.get('hpo_terms', []))}")

# =============================================================================
# STEP 5: Parse Fetal-Specific Sections (NEW)
# =============================================================================
print("\n[STEP 5] Parsing fetal-specific sections...")

# 5a. Fetal Anatomy (with HPO extraction from anomalies)
anatomy = parse_fetal_anatomy(observer_data, "observer_json", hpo_cr=hpo_cr)
print(f"\n  [5a] Fetal Anatomy:")
print(f"       Normal structures: {len(anatomy.get('normal_structures', []))}")
print(f"       Abnormal structures: {len(anatomy.get('abnormal_structures', []))}")
print(f"       Not visualized: {len(anatomy.get('not_visualized', []))}")
print(f"       Anomalies detected: {len(anatomy.get('anomalies', []))}")
print(f"       HPO terms extracted: {len(anatomy.get('hpo_terms', []))}")

for anomaly in anatomy.get("anomalies", [])[:3]:
   print(f"         o {anomaly.get('description', 'N/A')} ({anomaly.get('variant_type', 'N/A')})")

# 5b. Estimated Fetal Weight
efw = parse_estimated_fetal_weight(observer_data, "observer_json")
print(f"\n  [5b] Estimated Fetal Weight:")
print(f"       EFW: {efw.get('efw_grams', 'N/A')} grams")
print(f"       Percentile: {efw.get('percentile', 'N/A')}%")
print(f"       Method: {efw.get('method', 'N/A')}")
print(f"       Growth category: {efw.get('growth_category', 'N/A')}")
print(f"       Within normal range: {efw.get('within_normal_range', 'N/A')}")

# 5c. Fetal Ratios
ratios = parse_fetal_ratios(observer_data, "observer_json")
print(f"\n  [5c] Fetal Ratios:")
print(f"       Ratios calculated: {len(ratios.get('ratios', []))}")
print(f"       All within range: {ratios.get('all_within_range', 'N/A')}")
print(f"       Proportionality: {ratios.get('proportionality_assessment', 'N/A')}")

for ratio in ratios.get("ratios", [])[:3]:
   name = ratio.get("name", "N/A")
   value = ratio.get("value", "N/A")
   in_range = "[OK]" if ratio.get("within_range") else "[!]"
   print(f"         {in_range} {name}: {value}")

# =============================================================================
# STEP 6: Build PhenotypicFeatures from ALL sources
# =============================================================================
print("\n[STEP 6] Building PhenotypicFeatures...")

phenotypic_features = []

# Get subject GA from dating or fallback to first measurement
ga_weeks = dating.get("ga_weeks")
if ga_weeks:
   subject_ga = GestationalAge.from_weeks(float(ga_weeks))
else:
   # Fallback: parse from first TermBin description
   if term_bins:
       weeks, days = parse_ga_from_description(term_bins[0].description)
       subject_ga = GestationalAge(weeks=weeks, days=days)
   else:
       subject_ga = GestationalAge(weeks=27, days=0)

# 6a. From biometry TermBins
for tb in term_bins:
   weeks, days = parse_ga_from_description(tb.description)
   onset = pps2.TimeElement(
       gestational_age=pps2.GestationalAge(weeks=weeks, days=days)
   )
   pf = pps2.PhenotypicFeature(
       type=pps2.OntologyClass(id=tb.hpo_id, label=tb.hpo_label),
       excluded=tb.normal,  # normal=True means abnormality is EXCLUDED
       description=f"Biometry: {tb.description}",
       onset=onset,
   )
   phenotypic_features.append(("Biometry", pf))

# 6b. From clinical impression HPO terms (SimpleTerm objects with hpo_id, hpo_label)
for term in impression.get("hpo_terms", []):
   pf = pps2.PhenotypicFeature(
       type=pps2.OntologyClass(id=term.hpo_id, label=term.hpo_label),
       excluded=False,
       description=f"Clinical impression: {term.hpo_label}",
       onset=pps2.TimeElement(
           gestational_age=pps2.GestationalAge(weeks=subject_ga.weeks, days=subject_ga.days)
       ),
   )
   phenotypic_features.append(("Clinical Text", pf))

# 6c. From fetal anatomy HPO terms (SimpleTerm objects with hpo_id, hpo_label) (NEW)
for term in anatomy.get("hpo_terms", []):
   pf = pps2.PhenotypicFeature(
       type=pps2.OntologyClass(id=term.hpo_id, label=term.hpo_label),
       excluded=False,
       description=f"Anatomy finding: {term.hpo_label}",
       onset=pps2.TimeElement(
           gestational_age=pps2.GestationalAge(weeks=subject_ga.weeks, days=subject_ga.days)
       ),
   )
   phenotypic_features.append(("Anatomy", pf))

# 6d. Growth category as phenotypic feature (NEW)
growth_hpo_map = {
   "SGA": ("HP:0001518", "Small for gestational age"),
   "LGA": ("HP:0001520", "Large for gestational age"),
}
growth_cat = efw.get("growth_category")
if growth_cat in growth_hpo_map:
   hpo_id, hpo_label = growth_hpo_map[growth_cat]
   pf = pps2.PhenotypicFeature(
       type=pps2.OntologyClass(id=hpo_id, label=hpo_label),
       excluded=False,
       description=f"EFW {efw.get('efw_grams')}g at {efw.get('percentile')}th percentile",
       onset=pps2.TimeElement(
           gestational_age=pps2.GestationalAge(weeks=subject_ga.weeks, days=subject_ga.days)
       ),
   )
   phenotypic_features.append(("Growth", pf))
# AGA is normal - we could add as excluded feature or skip
elif growth_cat == "AGA":
   print("Growth category AGA (normal) - no HPO term needed")

# 6e. Proportionality assessment as phenotypic feature (NEW)
if ratios.get("proportionality_assessment") == "Asymmetric":
   pf = pps2.PhenotypicFeature(
       type=pps2.OntologyClass(id="HP:0001511", label="Intrauterine growth retardation"),
       excluded=False,
       description="Asymmetric growth pattern detected from biometric ratios",
       onset=pps2.TimeElement(
           gestational_age=pps2.GestationalAge(weeks=subject_ga.weeks, days=subject_ga.days)
       ),
   )
   phenotypic_features.append(("Ratios", pf))

print(f"\n  Summary by source:")
sources = {}
for source, pf in phenotypic_features:
   sources[source] = sources.get(source, 0) + 1
for source, count in sources.items():
   print(f"    - {source}: {count} features")
print(f"  Total: {len(phenotypic_features)} PhenotypicFeatures")

# =============================================================================
# STEP 7: Assemble Phenopacket v2.0
# =============================================================================
print("\n[STEP 7] Assembling Phenopacket v2.0...")

subject = pps2.Individual(
   id="fetus-1",
   sex=pps2.Sex.UNKNOWN_SEX,
   time_at_last_encounter=pps2.TimeElement(
       gestational_age=pps2.GestationalAge(weeks=subject_ga.weeks, days=subject_ga.days)
   ),
)

now = datetime.now(timezone.utc)
created_timestamp = Timestamp()
created_timestamp.FromDatetime(now)

hpo_resource = pps2.Resource(
   id="hp",
   name="Human Phenotype Ontology",
   url="http://purl.obolibrary.org/obo/hp.owl",
   version=hpo_parser.get_version() or "2025-01-01", # TODO (@VarenyaJ): Change version date if update the compressed hp.json
   namespace_prefix="HP",
   iri_prefix="http://purl.obolibrary.org/obo/HP_",
)

metadata = pps2.MetaData(
   created=created_timestamp,
   created_by="prenatalppkt-etl-pipeline-v2",
   phenopacket_schema_version="2.0",
)
metadata.resources.append(hpo_resource)

phenopacket = pps2.Phenopacket(
   id="apple-sally-fetus-1-complete",
   subject=subject,
   meta_data=metadata,
)
phenopacket.phenotypic_features.extend([pf for _, pf in phenotypic_features])

print(f"Phenopacket ID: {phenopacket.id}")
print(f"Subject: {phenopacket.subject.id}")
print(f"Features: {len(phenopacket.phenotypic_features)}")

# =============================================================================
# STEP 8: Output & Validation
# =============================================================================
print("\n[STEP 8] Output & Validation...")

phenopacket_json = MessageToJson(phenopacket, preserving_proto_field_name=True)

# Round-trip validation
parsed_back = Parse(phenopacket_json, pps2.Phenopacket())
assert parsed_back.id == phenopacket.id
assert len(parsed_back.phenotypic_features) == len(phenopacket.phenotypic_features)
print("Round-trip validation passed")

# Save to file
output_path = Path("notebook_outputs/initial_demo/apple_sally_phenopacket_complete.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, "w") as f:
   f.write(phenopacket_json)
print(f"Saved to: {output_path}")

# =============================================================================
# STEP 9: Summary Report
# =============================================================================
print("\n" + "=" * 80)
print("PHENOPACKET GENERATION COMPLETE")
print("=" * 80)

print("\n[Clinical Context]")
print(f"  Indication: {indication_text[:50]}...")
print(f"  GA at exam: {dating.get('ga_weeks', 'N/A')} weeks")
print(f"  EFW: {efw.get('efw_grams', 'N/A')}g ({efw.get('growth_category', 'N/A')})")
print(f"  Proportionality: {ratios.get('proportionality_assessment', 'N/A')}")

print("\n[Phenotypic Features by Source]")
for source, count in sources.items():
   print(f"  {source}: {count}")

observed = sum(1 for _, pf in phenotypic_features if not pf.excluded)
excluded = sum(1 for _, pf in phenotypic_features if pf.excluded)
print(f"\n[Feature Status]")
print(f"  Observed (abnormal): {observed}")
print(f"  Excluded (normal): {excluded}")

print("\n" + "=" * 80)
print(f"SUCCESS: Complete phenopacket at {output_path}")
print("=" * 80)

# =============================================================================
# STEP 10: Display JSON Output
# =============================================================================
print("\n[Phenopacket JSON Output]")
print(phenopacket_json)

# Basic Observer Process

In [ ]:
# =============================================================================
# BATCH: Generate phenopackets for all Observer JSON files in a directory
# Self-contained: all imports included so this cell can run independently
# =============================================================================

import gzip
import json
import re
from datetime import datetime, timezone
from pathlib import Path

from google.protobuf.json_format import MessageToJson, Parse
from google.protobuf.timestamp_pb2 import Timestamp
import phenopackets.schema.v2 as pps2

from prenatalppkt.etl.extractors import observer
from prenatalppkt.etl.sections import (
   parse_clinical_indication,
   parse_pregnancy_dating,
   parse_clinical_impression,
   parse_fetal_anatomy,
   parse_estimated_fetal_weight,
   parse_fetal_ratios,
)
from prenatalppkt.hpo import HpoParser
from prenatalppkt.gestational_age import GestationalAge

# -----------------------------------------------------------------------------
# HPO initialisation
# Reuse hpo_parser/hpo_cr if cell 1 has already been run (avoids a ~30s
# reload). Bootstrap from the compressed ontology file otherwise so this
# cell can be run standalone.
# -----------------------------------------------------------------------------
if "hpo_parser" not in dir():
   HP_JSON_GZ = Path("tests/data/hp.json.gz")
   TMP_HP_JSON = Path("/tmp/hp.json")
   with gzip.open(HP_JSON_GZ, "rt", encoding="utf-8") as f_in:
      with open(TMP_HP_JSON, "w", encoding="utf-8") as f_out:
         f_out.write(f_in.read())
   hpo_parser = HpoParser(hpo_json_file=str(TMP_HP_JSON))
   hpo_cr = hpo_parser.get_hpo_concept_recognizer()
   print(f"HPO loaded: {hpo_parser.get_version()}")
else:
   print(f"HPO already in scope: {hpo_parser.get_version()}")

# -----------------------------------------------------------------------------
# Helper: extract gestational age (weeks, days) from a TermBin description
# string, e.g. "HC: 250.0 mm (42.5%) at 26w6d" -> (26, 6).
# Redefined here so this cell works even if cell 1 hasn't been run.
# -----------------------------------------------------------------------------
def parse_ga_from_description(description: str) -> tuple[int, int]:
   """Extract weeks and days from TermBin description like 'HC: 250.0 mm (42.5%) at 26w6d'"""
   match = re.search(r"at (\d+)w(\d+)d", description)
   if match:
      return int(match.group(1)), int(match.group(2))
   return 27, 0  # fallback

# NOTE: T1 dispatch now lives in observer.extract() (PR vj/observer-t1-extractor).
# The old is_first_trimester_scan() guard below is disabled so T1 files (Diva) flow through.
# # -----------------------------------------------------------------------------
# # Helper: detect whether a fetus block contains only first-trimester
# # measurements (CRL, NT, etc.) and none of the T2/T3 biometry measurements
# # this pipeline requires (AC, BPD, HC, Femur).
# #
# # Background: first-trimester Observer JSON exports use different field names
# # ('label' instead of 'name', 'calculated_percentile' instead of
# # 'percentile') AND don't contain the required T2/T3 measurements at all.
# # Rather than raising a noisy error, we detect and skip these files early.
# # TODO (@VarenyaJ): implement a first-trimester ETL path (CRL -> HPO).
# # -----------------------------------------------------------------------------
# T1_ONLY_LABELS  = {"CRL", "NT"}
# T2T3_REQUIRED   = {"AC", "BPD", "HC", "Femur"}
#
# def is_first_trimester_scan(observer_data: dict) -> bool:
#    """Return True if the first fetus block contains only T1 measurements."""
#    fetus_measurements = (
#       observer_data.get("fetuses", [{}])[0].get("measurements", [])
#    )
#    # First-trimester exports use 'label'; standard exports use 'name'
#    labels = {
#       m.get("label") or m.get("name")
#       for m in fetus_measurements
#    }
#    has_t1_only = bool(labels & T1_ONLY_LABELS)
#    has_t2t3    = bool(labels & T2T3_REQUIRED)
#    return has_t1_only and not has_t2t3

# -----------------------------------------------------------------------------
# Batch configuration
# -----------------------------------------------------------------------------
DATA_DIR         = Path("tests/data")
BATCH_OUTPUT_DIR = Path("notebook_outputs/basic_observer_process")
BATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Only pick up Observer JSON exports; excludes hp.json.gz, HL7 files, etc.
json_files = sorted(DATA_DIR.glob("*_pretty.json"))

print("=" * 80)
print("BATCH PHENOPACKET GENERATION")
print(f"Directory : {DATA_DIR}")
print(f"Files     : {len(json_files)}")
print("=" * 80)

batch_results = []
batch_errors  = []
batch_skipped = []

for data_path in json_files:
   print(f"\n{'─' * 80}")
   print(f"Processing: {data_path.name}")
   print(f"{'─' * 80}")

   try:
      # ----------------------------------------------------------------------
      # Load raw Observer JSON
      # ----------------------------------------------------------------------
      with open(data_path) as f:
         observer_data = json.load(f)

      print(f"Fetuses: {len(observer_data.get('fetuses', []))}")

      # NOTE: guard disabled - observer.extract() now handles T1 scans directly.
#       # ----------------------------------------------------------------------
#       # Skip first-trimester scans: they lack T2/T3 biometry measurements
#       # (AC, BPD, HC, Femur) that the current ETL pipeline requires, and use
#       # different field names (label/calculated_percentile vs name/percentile).
#       # TODO (@VarenyaJ): implement a first-trimester ETL path (CRL -> HPO).
#       # ----------------------------------------------------------------------
#       if is_first_trimester_scan(observer_data):
#          print(f"  SKIP: First trimester scan (CRL/NT only) - T2/T3 pipeline not applicable")
#          batch_skipped.append({"file": data_path.name, "reason": "First trimester scan (CRL/NT only)"})
#          continue

      # ----------------------------------------------------------------------
      # Extract biometry -> List[TermBin]
      # Each TermBin holds: description, hpo_id, hpo_label, normal (bool)
      # normal=True means the measurement is within normal range, so the
      # corresponding HPO abnormality term is EXCLUDED in the phenopacket.
      # ----------------------------------------------------------------------
      file_term_bins = observer.extract(observer_data)
      print(f"\nExtracted {len(file_term_bins)} TermBins")
      for tb in file_term_bins:
         status = "Normal" if tb.normal else "ABNORMAL"
         print(f"    - {tb.description} [{status}]")
         print(f"      HPO: {tb.hpo_id} - {tb.hpo_label}")

      # ----------------------------------------------------------------------
      # Parse all clinical sections from the Observer JSON.
      # Each parser returns a plain dict; see etl/sections for field details.
      # ----------------------------------------------------------------------
      file_indication = parse_clinical_indication(observer_data, "observer_json")
      file_dating     = parse_pregnancy_dating(observer_data, "observer_json")
      file_impression = parse_clinical_impression(observer_data, "observer_json", hpo_cr=hpo_cr)
      file_anatomy    = parse_fetal_anatomy(observer_data, "observer_json", hpo_cr=hpo_cr)
      file_efw        = parse_estimated_fetal_weight(observer_data, "observer_json")
      file_ratios     = parse_fetal_ratios(observer_data, "observer_json")

      file_indication_text = file_indication.get("indication_text", "N/A") or "N/A"
      print(f"\n  Indication  : {file_indication_text[:60]}...")
      print(f"  GA at exam  : {file_dating.get('ga_weeks', 'N/A')} weeks")
      print(f"  EFW         : {file_efw.get('efw_grams', 'N/A')}g ({file_efw.get('growth_category', 'N/A')})")
      print(f"  HPO terms   : impression={len(file_impression.get('hpo_terms', []))}, anatomy={len(file_anatomy.get('hpo_terms', []))}")

      # ----------------------------------------------------------------------
      # Resolve subject gestational age for phenopacket onset timestamps.
      # Priority: (1) pregnancy dating section, (2) first TermBin description,
      # (3) hardcoded fallback of 27w0d.
      # ----------------------------------------------------------------------
      file_ga_weeks = file_dating.get("ga_weeks")
      if file_ga_weeks:
         file_subject_ga = GestationalAge.from_weeks(float(file_ga_weeks))
      elif file_term_bins:
         _weeks, _days = parse_ga_from_description(file_term_bins[0].description)
         file_subject_ga = GestationalAge(weeks=_weeks, days=_days)
      else:
         file_subject_ga = GestationalAge(weeks=27, days=0)

      # ----------------------------------------------------------------------
      # Build PhenotypicFeature list from all four data sources.
      # We track (source_label, PhenotypicFeature) tuples so we can report
      # counts by source in the summary.
      # ----------------------------------------------------------------------
      file_phenotypic_features = []

      # --- Source 1: Biometry TermBins ---
      # Each measurement's GA is taken from its own TermBin description (onset
      # varies per measurement since different structures are measured at
      # different gestational ages within the same exam).
      for tb in file_term_bins:
         weeks, days = parse_ga_from_description(tb.description)
         pf = pps2.PhenotypicFeature(
            type=pps2.OntologyClass(id=tb.hpo_id, label=tb.hpo_label),
            excluded=tb.normal,   # normal=True -> abnormality is EXCLUDED
            description=f"Biometry: {tb.description}",
            onset=pps2.TimeElement(
               gestational_age=pps2.GestationalAge(weeks=weeks, days=days)
            ),
         )
         file_phenotypic_features.append(("Biometry", pf))

      # --- Source 2: Clinical impression (free text -> HPO via fenominal) ---
      # SimpleTerm objects returned by fenominal carry
      # hpo_id and hpo_label attributes.
      for term in file_impression.get("hpo_terms", []):
         pf = pps2.PhenotypicFeature(
            type=pps2.OntologyClass(id=term.hpo_id, label=term.hpo_label),
            excluded=False,
            description=f"Clinical impression: {term.hpo_label}",
            onset=pps2.TimeElement(
               gestational_age=pps2.GestationalAge(weeks=file_subject_ga.weeks, days=file_subject_ga.days)
            ),
         )
         file_phenotypic_features.append(("Clinical Text", pf))

      # --- Source 3: Fetal anatomy section (structured anomaly findings -> HPO) ---
      for term in file_anatomy.get("hpo_terms", []):
         pf = pps2.PhenotypicFeature(
            type=pps2.OntologyClass(id=term.hpo_id, label=term.hpo_label),
            excluded=False,
            description=f"Anatomy finding: {term.hpo_label}",
            onset=pps2.TimeElement(
               gestational_age=pps2.GestationalAge(weeks=file_subject_ga.weeks, days=file_subject_ga.days)
            ),
         )
         file_phenotypic_features.append(("Anatomy", pf))

      # --- Source 4a: Growth category (SGA/LGA only) ---
      # AGA (appropriate for gestational age) is normal, so we omit it.
      # SGA and LGA are phenotypically significant and map directly to HPO.
      growth_hpo_map = {
         "SGA": ("HP:0001518", "Small for gestational age"),
         "LGA": ("HP:0001520", "Large for gestational age"),
      }
      file_growth_cat = file_efw.get("growth_category")
      if file_growth_cat in growth_hpo_map:
         hpo_id, hpo_label = growth_hpo_map[file_growth_cat]
         pf = pps2.PhenotypicFeature(
            type=pps2.OntologyClass(id=hpo_id, label=hpo_label),
            excluded=False,
            description=f"EFW {file_efw.get('efw_grams')}g at {file_efw.get('percentile')}th percentile",
            onset=pps2.TimeElement(
               gestational_age=pps2.GestationalAge(weeks=file_subject_ga.weeks, days=file_subject_ga.days)
            ),
         )
         file_phenotypic_features.append(("Growth", pf))
      elif file_growth_cat == "AGA":
         print("Growth category AGA (normal) - no HPO term needed")

      # --- Source 4b: Biometric proportionality (Asymmetric growth only) ---
      # Asymmetric IUGR (HC spared, AC reduced) maps to HP:0001511.
      # Normal proportionality produces no additional phenotypic feature.
      if file_ratios.get("proportionality_assessment") == "Asymmetric":
         pf = pps2.PhenotypicFeature(
            type=pps2.OntologyClass(id="HP:0001511", label="Intrauterine growth retardation"),
            excluded=False,
            description="Asymmetric growth pattern detected from biometric ratios",
            onset=pps2.TimeElement(
               gestational_age=pps2.GestationalAge(weeks=file_subject_ga.weeks, days=file_subject_ga.days)
            ),
         )
         file_phenotypic_features.append(("Ratios", pf))

      file_sources = {}
      for source, pf in file_phenotypic_features:
         file_sources[source] = file_sources.get(source, 0) + 1
      print(f"\n  Features by source: {dict(file_sources)}")
      print(f"  Total: {len(file_phenotypic_features)} PhenotypicFeatures")

      # ----------------------------------------------------------------------
      # Assemble GA4GH Phenopacket v2.0
      # ID convention: "<name>-fetus-1", e.g. "blue-sally-fetus-1"
      # (strips the "_pretty" export suffix and lowercases)
      # ----------------------------------------------------------------------
      file_stem = data_path.stem.replace("_pretty", "").replace("_", "-").lower()
      file_phenopacket_id = f"{file_stem}-fetus-1"

      file_now = datetime.now(timezone.utc)
      file_ts  = Timestamp()
      file_ts.FromDatetime(file_now)

      file_hpo_resource = pps2.Resource(
         id="hp",
         name="Human Phenotype Ontology",
         url="http://purl.obolibrary.org/obo/hp.owl",
         version=hpo_parser.get_version() or "2025-01-01",  # TODO (@VarenyaJ): Change version date if update the compressed hp.json
         namespace_prefix="HP",
         iri_prefix="http://purl.obolibrary.org/obo/HP_",
      )

      file_metadata = pps2.MetaData(
         created=file_ts,
         created_by="prenatalppkt-etl-pipeline-v2",
         phenopacket_schema_version="2.0",
      )
      file_metadata.resources.append(file_hpo_resource)

      file_phenopacket = pps2.Phenopacket(
         id=file_phenopacket_id,
         subject=pps2.Individual(
            id="fetus-1",
            sex=pps2.Sex.UNKNOWN_SEX,
            time_at_last_encounter=pps2.TimeElement(
               gestational_age=pps2.GestationalAge(weeks=file_subject_ga.weeks, days=file_subject_ga.days)
            ),
         ),
         meta_data=file_metadata,
      )
      file_phenopacket.phenotypic_features.extend([pf for _, pf in file_phenotypic_features])

      print(f"\nPhenopacket ID : {file_phenopacket.id}")
      print(f"Subject        : {file_phenopacket.subject.id}")
      print(f"Features       : {len(file_phenopacket.phenotypic_features)}")

      # ----------------------------------------------------------------------
      # Serialise, validate, and save
      # Round-trip validation: serialise to JSON then parse back and assert
      # ID and feature count are preserved.
      # ----------------------------------------------------------------------
      file_phenopacket_json = MessageToJson(file_phenopacket, preserving_proto_field_name=True)

      file_parsed_back = Parse(file_phenopacket_json, pps2.Phenopacket())
      assert file_parsed_back.id == file_phenopacket.id
      assert len(file_parsed_back.phenotypic_features) == len(file_phenopacket.phenotypic_features)
      print("Round-trip validation passed")

      file_output_path = BATCH_OUTPUT_DIR / f"{data_path.stem.lower().replace('_pretty', '_phenopacket')}.json"
      with open(file_output_path, "w") as f:
         f.write(file_phenopacket_json)
      print(f"Saved to: {file_output_path}")

      batch_results.append({
         "file": data_path.name,
         "phenopacket_id": file_phenopacket_id,
         "output_path": file_output_path,
         "n_features": len(file_phenotypic_features),
         "sources": file_sources,
         "ga": f"{file_subject_ga.weeks}w{file_subject_ga.days}d",
         "growth_category": file_growth_cat,
         "proportionality": file_ratios.get("proportionality_assessment"),
      })

   except Exception as e:
      print(f"  ERROR: {e}")
      batch_errors.append({"file": data_path.name, "error": str(e)})

# =============================================================================
# Batch Summary
# =============================================================================
print(f"\n{'=' * 80}")
print(f"BATCH COMPLETE  {len(batch_results)} succeeded  |  {len(batch_skipped)} skipped  |  {len(batch_errors)} failed  (of {len(json_files)} total)")
print(f"{'=' * 80}")

for r in batch_results:
   print(f"\n  {r['phenopacket_id']}")
   print(f"    Source file  : {r['file']}")
   print(f"    GA at exam   : {r['ga']}")
   print(f"    Growth       : {r['growth_category']}")
   print(f"    Proportional : {r['proportionality']}")
   print(f"    Features     : {r['n_features']}  {r['sources']}")
   print(f"    Saved to     : {r['output_path']}")

if batch_skipped:
   print(f"\n  SKIPPED ({len(batch_skipped)}):")
   for s in batch_skipped:
      print(f"    {s['file']}: {s['reason']}")

if batch_errors:
   print(f"\n  ERRORS ({len(batch_errors)}):")
   for e in batch_errors:
      print(f"    {e['file']}: {e['error']}")

# Biometry to HPO and LOINC

In [ ]:
# =============================================================================
# BATCH WITH LOINC MEASUREMENTS
# Loops every Observer JSON in tests/data and writes a GA4GH Phenopacket v2.0
# that carries BOTH phenotypicFeatures (HPO interpretations) and measurements
# (LOINC-coded raw biometry values). Self-contained: re-imports everything so
# this cell can run standalone.
# =============================================================================

import gzip
import json
import re
from datetime import datetime, timezone
from pathlib import Path

from google.protobuf.json_format import MessageToJson, Parse
from google.protobuf.timestamp_pb2 import Timestamp
import phenopackets.schema.v2 as pps2

from prenatalppkt.etl.extractors import observer
from prenatalppkt.etl.sections import (
   parse_clinical_indication,
   parse_pregnancy_dating,
   parse_clinical_impression,
   parse_fetal_anatomy,
   parse_estimated_fetal_weight,
   parse_fetal_ratios,
)
from prenatalppkt.hpo import HpoParser
from prenatalppkt.gestational_age import GestationalAge
from prenatalppkt.measurements.efw_measurement import build_efw_measurement

# -----------------------------------------------------------------------------
# HPO bootstrap
# Reuse hpo_parser/hpo_cr if an earlier cell already loaded them (the ontology
# parse takes ~30 seconds). Otherwise unzip the bundled hp.json.gz and parse
# it once. This keeps the cell runnable on its own without forcing the user
# to re-run the slow upstream load.
# -----------------------------------------------------------------------------
if "hpo_parser" not in dir():
   HP_JSON_GZ = Path("tests/data/hp.json.gz")
   TMP_HP_JSON = Path("/tmp/hp.json")
   with gzip.open(HP_JSON_GZ, "rt", encoding="utf-8") as f_in:
      with open(TMP_HP_JSON, "w", encoding="utf-8") as f_out:
         f_out.write(f_in.read())
   hpo_parser = HpoParser(hpo_json_file=str(TMP_HP_JSON))
   hpo_cr = hpo_parser.get_hpo_concept_recognizer()
   print(f"HPO loaded: {hpo_parser.get_version()}")
else:
   print(f"HPO already in scope: {hpo_parser.get_version()}")

# -----------------------------------------------------------------------------
# Helper: pull (weeks, days) out of a TermBin's description string.
# A TermBin description looks like "HC: 250.0 mm (42.5%) at 26w6d", and we
# need the 26 and 6 to stamp the right onset on each PhenotypicFeature.
# Used as a fallback when the pregnancy-dating section has no GA.
# -----------------------------------------------------------------------------
def parse_ga_from_description(description: str) -> tuple[int, int]:
   """Extract (weeks, days) from a TermBin description."""
   match = re.search(r"at (\d+)w(\d+)d", description)
   if match:
      return int(match.group(1)), int(match.group(2))
   return 27, 0  # fallback

# -----------------------------------------------------------------------------
# Helper: skip first-trimester scans early.
# First-trimester Observer exports only carry CRL and NT. The current
# pipeline expects the T2/T3 biometry set (AC, BPD, HC, Femur) and will
# raise if those are missing. Detecting the T1 shape up front lets us emit
# a clean SKIP message instead of a noisy error.
# TODO (@VarenyaJ): wire a first-trimester ETL path (CRL -> HPO term).
# -----------------------------------------------------------------------------
T1_ONLY_LABELS = {"CRL", "NT"}
T2T3_REQUIRED = {"AC", "BPD", "HC", "Femur"}

def is_first_trimester_scan(observer_data: dict) -> bool:
   """Return True when the first fetus has only T1 measurements (no T2/T3)."""
   fetus_measurements = (
      observer_data.get("fetuses", [{}])[0].get("measurements", [])
   )
   # Some exports use 'label', others use 'name'; accept either.
   labels = {m.get("label") or m.get("name") for m in fetus_measurements}
   has_t1_only = bool(labels & T1_ONLY_LABELS)
   has_t2t3 = bool(labels & T2T3_REQUIRED)
   return has_t1_only and not has_t2t3

# -----------------------------------------------------------------------------
# Batch configuration
# tests/data/*json picks up every JSON file in the test corpus. Today they
# are all *_pretty.json, but the wider glob covers future exports too (and
# safely skips hp.json.gz since it ends in .gz, not .json).
# outputs/notebook_phenopackets/ is gitignored alongside output/.
# -----------------------------------------------------------------------------
DATA_DIR = Path("tests/data")
BATCH_OUTPUT_DIR = Path("notebook_outputs/biometry_to_hpo_and_loinc")
BATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

json_files = sorted(DATA_DIR.glob("*json"))

print("=" * 80)
print("BATCH PHENOPACKET GENERATION WITH LOINC MEASUREMENTS")
print(f"Directory : {DATA_DIR}")
print(f"Output    : {BATCH_OUTPUT_DIR}")
print(f"Files     : {len(json_files)}")
print("=" * 80)

batch_results = []
batch_errors = []
batch_skipped = []

for data_path in json_files:
   print(f"\n{'─' * 80}")
   print(f"Processing: {data_path.name}")
   print(f"{'─' * 80}")

   try:
      # ----------------------------------------------------------------------
      # Load the raw Observer JSON.
      # ----------------------------------------------------------------------
      with open(data_path) as f:
         observer_data = json.load(f)

      print(f"Fetuses: {len(observer_data.get('fetuses', []))}")

      # ----------------------------------------------------------------------
      # Skip first-trimester scans (CRL/NT only). The current pipeline needs
      # T2/T3 biometry; revisit when the T1 path lands.
      # ----------------------------------------------------------------------
      if is_first_trimester_scan(observer_data):
         print("  SKIP: First trimester scan (CRL/NT only) - T2/T3 pipeline not applicable")
         batch_skipped.append({
            "file": data_path.name,
            "reason": "First trimester scan (CRL/NT only)",
         })
         continue

      # ----------------------------------------------------------------------
      # Extract biometry into TermBins.
      # Each TermBin now carries both the HPO interpretation (hpo_id,
      # hpo_label, normal) AND the raw LOINC context (loinc_code, value_mm,
      # gestational_age_weeks). The HPO side drives phenotypicFeatures; the
      # LOINC side drives the new measurements array.
      # ----------------------------------------------------------------------
      file_term_bins = observer.extract(observer_data)
      print(f"\nExtracted {len(file_term_bins)} TermBins")
      for tb in file_term_bins:
         status = "Normal" if tb.normal else "ABNORMAL"
         loinc = tb.loinc_code or "(no LOINC)"
         print(f"    - {tb.description} [{status}]")
         print(f"      HPO   : {tb.hpo_id} - {tb.hpo_label}")
         print(f"      LOINC : {loinc}")

      # ----------------------------------------------------------------------
      # Parse every clinical section. Each parser returns a plain dict; see
      # etl/sections for field names.
      # ----------------------------------------------------------------------
      file_indication = parse_clinical_indication(observer_data, "observer_json")
      file_dating = parse_pregnancy_dating(observer_data, "observer_json")
      file_impression = parse_clinical_impression(observer_data, "observer_json", hpo_cr=hpo_cr)
      file_anatomy = parse_fetal_anatomy(observer_data, "observer_json", hpo_cr=hpo_cr)
      file_efw = parse_estimated_fetal_weight(observer_data, "observer_json")
      file_ratios = parse_fetal_ratios(observer_data, "observer_json")

      file_indication_text = file_indication.get("indication_text", "N/A") or "N/A"
      print(f"\n  Indication  : {file_indication_text[:60]}...")
      print(f"  GA at exam  : {file_dating.get('ga_weeks', 'N/A')} weeks")
      print(f"  EFW         : {file_efw.get('efw_grams', 'N/A')}g ({file_efw.get('growth_category', 'N/A')})")
      print(f"  HPO terms   : impression={len(file_impression.get('hpo_terms', []))}, anatomy={len(file_anatomy.get('hpo_terms', []))}")

      # ----------------------------------------------------------------------
      # Pick a subject gestational age for the phenopacket's onset stamps.
      # Order: (1) pregnancy-dating section, (2) first TermBin description,
      # (3) hardcoded 27w0d fallback.
      # ----------------------------------------------------------------------
      file_ga_weeks = file_dating.get("ga_weeks")
      if file_ga_weeks:
         file_subject_ga = GestationalAge.from_weeks(float(file_ga_weeks))
      elif file_term_bins:
         _weeks, _days = parse_ga_from_description(file_term_bins[0].description)
         file_subject_ga = GestationalAge(weeks=_weeks, days=_days)
      else:
         file_subject_ga = GestationalAge(weeks=27, days=0)

      # ----------------------------------------------------------------------
      # Build the PhenotypicFeature list from all four sources. Tracking the
      # source label alongside each feature lets us report counts per source
      # in the summary at the end.
      # ----------------------------------------------------------------------
      file_phenotypic_features = []

      # --- Source 1: Biometry TermBins ---
      # Per-measurement onset GA: each TermBin's description carries the GA
      # at which that structure was measured, so onset can vary within one
      # exam (e.g. HC measured at 26w6d, AC measured at 26w3d).
      for tb in file_term_bins:
         weeks, days = parse_ga_from_description(tb.description)
         pf = pps2.PhenotypicFeature(
            type=pps2.OntologyClass(id=tb.hpo_id, label=tb.hpo_label),
            excluded=tb.normal,   # normal=True -> the HPO abnormality is EXCLUDED
            description=f"Biometry: {tb.description}",
            onset=pps2.TimeElement(
               gestational_age=pps2.GestationalAge(weeks=weeks, days=days)
            ),
         )
         file_phenotypic_features.append(("Biometry", pf))

      # --- Source 2: Clinical impression text -> HPO via fenominal ---
      # The recognizer returns SimpleTerm objects (hpo_id, hpo_label).
      for term in file_impression.get("hpo_terms", []):
         pf = pps2.PhenotypicFeature(
            type=pps2.OntologyClass(id=term.hpo_id, label=term.hpo_label),
            excluded=False,
            description=f"Clinical impression: {term.hpo_label}",
            onset=pps2.TimeElement(
               gestational_age=pps2.GestationalAge(weeks=file_subject_ga.weeks, days=file_subject_ga.days)
            ),
         )
         file_phenotypic_features.append(("Clinical Text", pf))

      # --- Source 3: Fetal anatomy section (structured anomaly findings) ---
      for term in file_anatomy.get("hpo_terms", []):
         pf = pps2.PhenotypicFeature(
            type=pps2.OntologyClass(id=term.hpo_id, label=term.hpo_label),
            excluded=False,
            description=f"Anatomy finding: {term.hpo_label}",
            onset=pps2.TimeElement(
               gestational_age=pps2.GestationalAge(weeks=file_subject_ga.weeks, days=file_subject_ga.days)
            ),
         )
         file_phenotypic_features.append(("Anatomy", pf))

      # --- Source 4a: Growth category (SGA/LGA only) ---
      # AGA = appropriate for gestational age = normal, so it adds no HPO term.
      growth_hpo_map = {
         "SGA": ("HP:0001518", "Small for gestational age"),
         "LGA": ("HP:0001520", "Large for gestational age"),
      }
      file_growth_cat = file_efw.get("growth_category")
      if file_growth_cat in growth_hpo_map:
         hpo_id, hpo_label = growth_hpo_map[file_growth_cat]
         pf = pps2.PhenotypicFeature(
            type=pps2.OntologyClass(id=hpo_id, label=hpo_label),
            excluded=False,
            description=f"EFW {file_efw.get('efw_grams')}g at {file_efw.get('percentile')}th percentile",
            onset=pps2.TimeElement(
               gestational_age=pps2.GestationalAge(weeks=file_subject_ga.weeks, days=file_subject_ga.days)
            ),
         )
         file_phenotypic_features.append(("Growth", pf))
      elif file_growth_cat == "AGA":
         print("Growth category AGA (normal) - no HPO term needed")

      # --- Source 4b: Asymmetric biometric ratios -> IUGR ---
      # Asymmetric IUGR (head spared, abdomen lagging) maps to HP:0001511.
      if file_ratios.get("proportionality_assessment") == "Asymmetric":
         pf = pps2.PhenotypicFeature(
            type=pps2.OntologyClass(id="HP:0001511", label="Intrauterine growth retardation"),
            excluded=False,
            description="Asymmetric growth pattern detected from biometric ratios",
            onset=pps2.TimeElement(
               gestational_age=pps2.GestationalAge(weeks=file_subject_ga.weeks, days=file_subject_ga.days)
            ),
         )
         file_phenotypic_features.append(("Ratios", pf))

      file_sources = {}
      for source, pf in file_phenotypic_features:
         file_sources[source] = file_sources.get(source, 0) + 1
      print(f"\n  Features by source: {dict(file_sources)}")
      print(f"  Total: {len(file_phenotypic_features)} PhenotypicFeatures")

      # ----------------------------------------------------------------------
      # *** NEW *** Build LOINC measurements from each TermBin.
      # PhenotypicFeatures answer "what abnormality is present?" (HPO term);
      # Measurements answer "what was the raw observation?" (LOINC code + mm).
      # Both refer to the same scan: an Observer JSON entry for, say, HC
      # produces one PhenotypicFeature (the HPO interpretation) AND one
      # Measurement (LOINC:11984-2 with the raw mm value).
      # TermBins without a LOINC code (e.g. OFD - no verified LOINC yet) are
      # skipped cleanly via the None guard.
      # ----------------------------------------------------------------------
      file_measurements = []
      for tb in file_term_bins:
         if tb.loinc_code is None or tb.value_mm is None:
            continue
         msmt = pps2.Measurement(
            assay=pps2.OntologyClass(id=tb.loinc_code, label=tb.loinc_label),
            value=pps2.Value(
               quantity=pps2.Quantity(
                  unit=pps2.OntologyClass(id="UO:0000016", label="millimeter"),
                  value=tb.value_mm,
               )
            ),
         )
         file_measurements.append(msmt)
      print(f"  Total: {len(file_measurements)} Measurements (LOINC-coded biometry)")

      # ----------------------------------------------------------------------
      # *** NEW *** Build the EFW Measurement.
      # EFW is calculated from biometry (Hadlock formula etc.), not measured
      # directly, so it sits outside the TermBin pipeline. The section parser
      # already produced `file_efw` with `efw_grams`. We turn that into a
      # LOINC:11727-5 Measurement with unit UO:0000021 (grams).
      # ----------------------------------------------------------------------
      efw_dict = build_efw_measurement(file_efw)
      if efw_dict is not None:
         efw_msmt = pps2.Measurement(
            assay=pps2.OntologyClass(id=efw_dict["assay"]["id"], label=efw_dict["assay"]["label"]),
            value=pps2.Value(
               quantity=pps2.Quantity(
                  unit=pps2.OntologyClass(id=efw_dict["value"]["quantity"]["unit"]["id"], label=efw_dict["value"]["quantity"]["unit"]["label"]),
                  value=efw_dict["value"]["quantity"]["value"],
               )
            ),
         )
         file_measurements.append(efw_msmt)
         _efw_grams = efw_dict["value"]["quantity"]["value"]
         print(f"  +1 EFW Measurement (LOINC:11727-5, {_efw_grams}g)")
      else:
         print("  No EFW Measurement (efw_grams missing)")

      # ----------------------------------------------------------------------
      # Assemble the GA4GH Phenopacket v2.0.
      # ID convention: "<file-stem-without-_pretty>-fetus-1" (lowercased).
      # When measurements are present we also list LOINC as a referenced
      # resource so the assay codes resolve.
      # ----------------------------------------------------------------------
      file_stem = data_path.stem.replace("_pretty", "").replace("_", "-").lower()
      file_phenopacket_id = f"{file_stem}-fetus-1"

      file_now = datetime.now(timezone.utc)
      file_ts = Timestamp()
      file_ts.FromDatetime(file_now)

      file_hpo_resource = pps2.Resource(
         id="hp",
         name="Human Phenotype Ontology",
         url="http://purl.obolibrary.org/obo/hp.owl",
         version=hpo_parser.get_version() or "2025-01-01",   # TODO (@VarenyaJ): refresh when hp.json updates
         namespace_prefix="HP",
         iri_prefix="http://purl.obolibrary.org/obo/HP_",
      )

      file_loinc_resource = pps2.Resource(
         id="loinc",
         name="Logical Observation Identifiers Names and Codes",
         url="https://loinc.org",
         version="2.78",                                      # TODO (@VarenyaJ): pin the LOINC release used to verify codes
         namespace_prefix="LOINC",
         iri_prefix="https://loinc.org/",
      )

      file_uo_resource = pps2.Resource(
         id="uo",
         name="Units of Measurement Ontology",
         url="http://purl.obolibrary.org/obo/uo.owl",
         version="2023-05-25",
         namespace_prefix="UO",
         iri_prefix="http://purl.obolibrary.org/obo/UO_",
      )

      file_metadata = pps2.MetaData(
         created=file_ts,
         created_by="prenatalppkt-etl-pipeline-v2",
         phenopacket_schema_version="2.0",
      )
      file_metadata.resources.append(file_hpo_resource)
      if file_measurements:
         file_metadata.resources.append(file_loinc_resource)
         file_metadata.resources.append(file_uo_resource)

      file_phenopacket = pps2.Phenopacket(
         id=file_phenopacket_id,
         subject=pps2.Individual(
            id="fetus-1",
            sex=pps2.Sex.UNKNOWN_SEX,
            time_at_last_encounter=pps2.TimeElement(
               gestational_age=pps2.GestationalAge(weeks=file_subject_ga.weeks, days=file_subject_ga.days)
            ),
         ),
         meta_data=file_metadata,
      )
      file_phenopacket.phenotypic_features.extend([pf for _, pf in file_phenotypic_features])
      file_phenopacket.measurements.extend(file_measurements)

      print(f"\nPhenopacket ID : {file_phenopacket.id}")
      print(f"Subject        : {file_phenopacket.subject.id}")
      print(f"Features       : {len(file_phenopacket.phenotypic_features)}")
      print(f"Measurements   : {len(file_phenopacket.measurements)}")

      # ----------------------------------------------------------------------
      # Serialise, validate, and save.
      # Round-trip validation: serialise to JSON, parse it back, and assert
      # the ID + feature count + measurement count all survived. Catches
      # subtle proto mistakes (wrong field name, missing oneof, etc.).
      # ----------------------------------------------------------------------
      file_phenopacket_json = MessageToJson(file_phenopacket, preserving_proto_field_name=True)

      file_parsed_back = Parse(file_phenopacket_json, pps2.Phenopacket())
      assert file_parsed_back.id == file_phenopacket.id
      assert len(file_parsed_back.phenotypic_features) == len(file_phenopacket.phenotypic_features)
      assert len(file_parsed_back.measurements) == len(file_phenopacket.measurements)
      print("Round-trip validation passed")

      file_output_path = BATCH_OUTPUT_DIR / f"{data_path.stem.lower().replace('_pretty', '_phenopacket')}.json"
      with open(file_output_path, "w") as f:
         f.write(file_phenopacket_json)
      print(f"Saved to: {file_output_path}")

      batch_results.append({
         "file": data_path.name,
         "phenopacket_id": file_phenopacket_id,
         "output_path": file_output_path,
         "n_features": len(file_phenotypic_features),
         "n_measurements": len(file_measurements),
         "sources": file_sources,
         "ga": f"{file_subject_ga.weeks}w{file_subject_ga.days}d",
         "growth_category": file_growth_cat,
         "proportionality": file_ratios.get("proportionality_assessment"),
      })

   except Exception as e:
      print(f"  ERROR: {e}")
      batch_errors.append({"file": data_path.name, "error": str(e)})

# =============================================================================
# Batch summary
# Reports per-file totals for both PhenotypicFeatures and Measurements.
# =============================================================================
print(f"\n{'=' * 80}")
print(f"BATCH COMPLETE  {len(batch_results)} succeeded  |  {len(batch_skipped)} skipped  |  {len(batch_errors)} failed  (of {len(json_files)} total)")
print(f"{'=' * 80}")

for r in batch_results:
   print(f"\n  {r['phenopacket_id']}")
   print(f"    Source file  : {r['file']}")
   print(f"    GA at exam   : {r['ga']}")
   print(f"    Growth       : {r['growth_category']}")
   print(f"    Proportional : {r['proportionality']}")
   print(f"    Features     : {r['n_features']}  {r['sources']}")
   print(f"    Measurements : {r['n_measurements']}  (LOINC-coded raw biometry)")
   print(f"    Saved to     : {r['output_path']}")

if batch_skipped:
   print(f"\n  SKIPPED ({len(batch_skipped)}):")
   for s in batch_skipped:
      print(f"    {s['file']}: {s['reason']}")

if batch_errors:
   print(f"\n  ERRORS ({len(batch_errors)}):")
   for e in batch_errors:
      print(f"    {e['file']}: {e['error']}")


# ViewPoint HL7 to HPO and LOINC

In [ ]:
# =============================================================================
# BATCH: ViewPoint HL7 -> Phenopacket (uses build_viewpoint_phenopacket)
# Loops every ViewPoint HL7 fixture in tests/data/, builds the core
# Phenopacket via the real build_viewpoint_phenopacket() builder (biometry +
# clinical impression + fetal anatomy PhenotypicFeatures, subject/id, HPO
# dedup), then adds LOINC-coded Measurement objects on top - mirroring the
# "Biometry to HPO and LOINC" cell above, but for the ViewPoint source
# instead of Observer. Self-contained: all imports included so this cell
# can run independently.
#
# Estimated fetal weight and fetal ratios are still skeleton parsers for
# viewpoint_hl7 - called here for completeness and reported as N/A, matching
# how the Observer cells above already report gaps honestly rather than
# hiding them.
# =============================================================================

import gzip
from datetime import datetime, timezone
from pathlib import Path

from google.protobuf.json_format import MessageToJson, Parse
from google.protobuf.timestamp_pb2 import Timestamp
import phenopackets.schema.v2 as pps2

from prenatalppkt.builders import build_viewpoint_phenopacket
from prenatalppkt.etl.extractors import viewpoint_hl7
from prenatalppkt.etl.sections import (
    parse_clinical_indication,
    parse_pregnancy_dating,
    parse_estimated_fetal_weight,
    parse_fetal_ratios,
)
from prenatalppkt.hpo import HpoParser

if "hpo_parser" not in dir():
    HP_JSON_GZ = Path("tests/data/hp.json.gz")
    TMP_HP_JSON = Path("/tmp/hp.json")
    with gzip.open(HP_JSON_GZ, "rt", encoding="utf-8") as f_in:
        with open(TMP_HP_JSON, "w", encoding="utf-8") as f_out:
            f_out.write(f_in.read())
    hpo_parser = HpoParser(hpo_json_file=str(TMP_HP_JSON))
    print(f"HPO loaded: {hpo_parser.get_version()}")
else:
    print(f"HPO already in scope: {hpo_parser.get_version()}")

DATA_DIR = Path("tests/data")
BATCH_OUTPUT_DIR = Path("notebook_outputs/viewpoint_hl7_to_hpo_and_loinc")
BATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

hl7_files = sorted(DATA_DIR.glob("viewpoint_hl7*.txt"))

ACCESSION_BY_FILE = {
    "viewpoint_hl7_test.txt": "DEMO001",
    "viewpoint_hl7_twins_test.txt": "DEMOTWIN",
    "viewpoint_hl7_full_exam_test.txt": "DEMOFULL",
    "viewpoint_hl7_anatomy_test.txt": "DEMOANAT",
}

print("=" * 80)
print("BATCH PHENOPACKET GENERATION - VIEWPOINT HL7")
print(f"Directory : {DATA_DIR}")
print(f"Output    : {BATCH_OUTPUT_DIR}")
print(f"Files     : {len(hl7_files)}")
print("=" * 80)

batch_results = []
batch_errors = []
batch_empty = []

for hl7_path in hl7_files:
    print(f"\n{'-' * 80}")
    print(f"Processing: {hl7_path.name}")
    print(f"{'-' * 80}")

    try:
        hl7_data = hl7_path.read_text()
        accession_id = ACCESSION_BY_FILE.get(hl7_path.name, "DEMO")

        file_ts = Timestamp()
        file_ts.FromDatetime(datetime.now(timezone.utc))

        pps = build_viewpoint_phenopacket(
            hl7_data, hpo_parser, file_ts, accession_id=accession_id
        )

        if not pps:
            print("  No fetuses found (no biometry OBX segments) - skipping")
            batch_empty.append(
                {"file": hl7_path.name, "reason": "No biometry OBX segments"}
            )
            continue

        indication = parse_clinical_indication(hl7_data, "viewpoint_hl7")
        dating = parse_pregnancy_dating(hl7_data, "viewpoint_hl7")
        efw = parse_estimated_fetal_weight(hl7_data, "viewpoint_hl7")
        ratios = parse_fetal_ratios(hl7_data, "viewpoint_hl7")

        indication_text = indication.get("indication_text", "N/A") or "N/A"
        print(f"  Indication  : {indication_text[:60]}")
        print(f"  GA by LMP   : {dating.get('ga_by_lmp', 'N/A')}")
        print(f"  EFW         : {efw.get('efw_grams', 'N/A')} (skeleton for viewpoint_hl7)")
        print(
            f"  Ratios      : {ratios.get('proportionality_assessment', 'N/A')} "
            "(skeleton for viewpoint_hl7)"
        )

        bins_by_fetus = viewpoint_hl7.extract_all_fetuses(hl7_data)

        for pp in pps:
            fetus_number = int(pp.subject.id.rsplit("-", 1)[-1])
            term_bins = bins_by_fetus.get(fetus_number, [])
            measurements = []
            for tb in term_bins:
                if tb.loinc_code is None or tb.value_mm is None:
                    continue
                measurements.append(
                    pps2.Measurement(
                        assay=pps2.OntologyClass(id=tb.loinc_code, label=tb.loinc_label),
                        value=pps2.Value(
                            quantity=pps2.Quantity(
                                unit=pps2.OntologyClass(id="UO:0000016", label="millimeter"),
                                value=tb.value_mm,
                            )
                        ),
                    )
                )
            pp.measurements.extend(measurements)
            if measurements:
                pp.meta_data.resources.append(
                    pps2.Resource(
                        id="loinc",
                        name="Logical Observation Identifiers Names and Codes",
                        url="https://loinc.org",
                        version="2.78",
                        namespace_prefix="LOINC",
                        iri_prefix="https://loinc.org/",
                    )
                )
                pp.meta_data.resources.append(
                    pps2.Resource(
                        id="uo",
                        name="Units of Measurement Ontology",
                        url="http://purl.obolibrary.org/obo/uo.owl",
                        version="2023-05-25",
                        namespace_prefix="UO",
                        iri_prefix="http://purl.obolibrary.org/obo/UO_",
                    )
                )

            print(f"\n  Phenopacket ID : {pp.id}")
            print(f"  Subject        : {pp.subject.id}")
            print(f"  Features       : {len(pp.phenotypic_features)}")
            print(f"  Measurements   : {len(pp.measurements)}")

            pp_json = MessageToJson(pp, preserving_proto_field_name=True)
            parsed_back = Parse(pp_json, pps2.Phenopacket())
            assert parsed_back.id == pp.id
            assert len(parsed_back.phenotypic_features) == len(pp.phenotypic_features)
            assert len(parsed_back.measurements) == len(pp.measurements)

            out_path = BATCH_OUTPUT_DIR / f"{pp.id}_phenopacket.json"
            with open(out_path, "w") as f:
                f.write(pp_json)
            print(f"  Saved to       : {out_path}")

            batch_results.append(
                {
                    "file": hl7_path.name,
                    "phenopacket_id": pp.id,
                    "output_path": out_path,
                    "n_features": len(pp.phenotypic_features),
                    "n_measurements": len(pp.measurements),
                }
            )

    except Exception as e:
        print(f"  ERROR: {e}")
        batch_errors.append({"file": hl7_path.name, "error": str(e)})

print(f"\n{'=' * 80}")
print(
    f"BATCH COMPLETE  {len(batch_results)} phenopackets  |  {len(batch_empty)} empty  |  "
    f"{len(batch_errors)} failed  (from {len(hl7_files)} files)"
)
print(f"{'=' * 80}")

for r in batch_results:
    print(f"\n  {r['phenopacket_id']}")
    print(f"    Source file  : {r['file']}")
    print(f"    Features     : {r['n_features']}")
    print(f"    Measurements : {r['n_measurements']}  (LOINC-coded raw biometry)")
    print(f"    Saved to     : {r['output_path']}")

if batch_empty:
    print(f"\n  EMPTY ({len(batch_empty)}):")
    for s in batch_empty:
        print(f"    {s['file']}: {s['reason']}")

if batch_errors:
    print(f"\n  ERRORS ({len(batch_errors)}):")
    for e in batch_errors:
        print(f"    {e['file']}: {e['error']}")


# Genomics Scaffold: Attach VCF Variants to a Phenopacket

In [ ]:
# =============================================================================
# Genomics scaffold demo
# Scan a synthetic VCF, strip it to PHI-safe loci (FORMAT + per-sample genotype
# columns are dropped on read), and attach the variants to a Phenopacket as:
#   (a) a files-by-URI File entry, and
#   (b) an inert Interpretation (VariationDescriptor.vcf_record).
# No VRS normalisation, no ACMG calls - structure only. Self-contained.
# =============================================================================

from pathlib import Path

import phenopackets.schema.v2 as pps2
from google.protobuf.json_format import MessageToJson

from prenatalppkt.genomics import (
   build_genomic_interpretation,
   build_vcf_file_entry,
   scan_vcf_file,
)

GENO_DATA_DIR = Path("tests/data")
vcf_path = GENO_DATA_DIR / "Apple_Sally.vcf"

variants = scan_vcf_file(vcf_path)
print(
   f"Scanned {len(variants)} variant loci from {vcf_path.name} "
   f"(FORMAT + sample genotype columns stripped)"
)
for v in variants:
   print(f"    {v.chrom}:{v.pos} {v.ref}>{v.alt}  ({v.genome_assembly})")

# Reuse the phenopacket built by the batch cell above when present; otherwise
# build a fresh demo Phenopacket so this cell can also run standalone.
geno_pp = (
   file_phenopacket
   if "file_phenopacket" in dir()
   else pps2.Phenopacket(id="apple-sally-genomics-demo")
)

geno_pp.files.append(
   build_vcf_file_entry(
      vcf_path.resolve().as_uri(),
      attributes={"genomeAssembly": variants[0].genome_assembly},
   )
)
geno_pp.interpretations.append(
   build_genomic_interpretation(
      variants,
      subject_id=geno_pp.subject.id or "fetus-1",
      interpretation_id=f"{geno_pp.id}-genomic-interp-1",
   )
)

print(
   f"\nPhenopacket '{geno_pp.id}' now carries "
   f"{len(geno_pp.files)} file(s) and {len(geno_pp.interpretations)} interpretation(s)"
)
print("\n[Genomic sections JSON]")
print(MessageToJson(geno_pp, preserving_proto_field_name=True))